In [ ]:
# ==========================================================
# TASK 4: PORTFOLIO OPTIMIZATION USING MODERN PORTFOLIO THEORY
# TSLA + SPY + BND
# ==========================================================


# -----------------------------
# Import Libraries
# -----------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import yfinance as yf



# ==========================================================
# 1. Load TSLA Forecast From Task 3
# ==========================================================


future_forecast = pd.read_csv(
    "tsla_future_forecast.csv",
    index_col=0,
    parse_dates=True
)


print("TSLA forecast loaded")

display(
    future_forecast.head()
)




# ==========================================================
# 2. Download Historical Market Data
# ==========================================================


assets = [
    "TSLA",
    "SPY",
    "BND"
]


data = yf.download(
    assets,
    start="2020-01-01",
    end="2026-01-01",
    auto_adjust=False
)



prices = data["Close"]


prices = prices.dropna()



print("\nHistorical Prices")

display(
    prices.head()
)





# ==========================================================
# 3. Calculate Daily Returns
# ==========================================================


returns = prices.pct_change().dropna()



print("\nDaily Returns")

display(
    returns.head()
)





# ==========================================================
# 4. Expected Returns
# ==========================================================


# -----------------------------
# TSLA Forecast Expected Return
# -----------------------------


current_tsla_price = prices["TSLA"].iloc[-1]


future_tsla_price = future_forecast["Forecast"].iloc[-1]



tsla_expected_return = (

    future_tsla_price -
    current_tsla_price

) / current_tsla_price




print("Current TSLA Price:",
      round(current_tsla_price,2))


print("Forecast TSLA Price:",
      round(future_tsla_price,2))


print("TSLA Forecast Return:",
      round(tsla_expected_return*100,2),
      "%")





# -----------------------------
# SPY and BND Historical Returns
# -----------------------------


spy_expected_return = (
    returns["SPY"].mean()
    *
    252
)



bnd_expected_return = (
    returns["BND"].mean()
    *
    252
)





expected_returns = np.array(
    [
        tsla_expected_return,
        spy_expected_return,
        bnd_expected_return
    ]
)



print("\nAnnual Expected Returns")

display(
    pd.Series(
        expected_returns,
        index=[
            "TSLA",
            "SPY",
            "BND"
        ]
    )
)





# ==========================================================
# 5. Covariance Matrix
# ==========================================================


cov_matrix = returns.cov() * 252



print("\nAnnualized Covariance Matrix")

display(
    cov_matrix
)





# Covariance Heatmap


plt.figure(
    figsize=(7,5)
)


sns.heatmap(
    cov_matrix,
    annot=True,
    cmap="coolwarm"
)


plt.title(
    "Annualized Covariance Matrix"
)


plt.show()





# ==========================================================
# 6. Portfolio Functions
# ==========================================================


def portfolio_return(weights):

    return np.dot(
        weights,
        expected_returns
    )




def portfolio_volatility(weights):

    return np.sqrt(
        np.dot(
            weights.T,
            np.dot(
                cov_matrix.values,
                weights
            )
        )
    )





def portfolio_sharpe(weights):

    return (
        portfolio_return(weights)
        /
        portfolio_volatility(weights)
    )





# ==========================================================
# 7. Efficient Frontier Simulation
# ==========================================================


num_portfolios = 10000


portfolio_results = []

weights_list = []



for i in range(num_portfolios):


    weights = np.random.random(3)


    weights = weights / np.sum(weights)



    ret = portfolio_return(
        weights
    )


    vol = portfolio_volatility(
        weights
    )


    sharpe = ret / vol



    portfolio_results.append(
        [
            ret,
            vol,
            sharpe
        ]
    )


    weights_list.append(
        weights
    )





portfolio_results = pd.DataFrame(

    portfolio_results,

    columns=[
        "Return",
        "Volatility",
        "Sharpe"
    ]

)





# ==========================================================
# 8. Find Optimal Portfolios
# ==========================================================


max_sharpe_index = (
    portfolio_results["Sharpe"]
    .idxmax()
)



min_vol_index = (
    portfolio_results["Volatility"]
    .idxmin()
)




max_sharpe = portfolio_results.loc[
    max_sharpe_index
]


min_vol = portfolio_results.loc[
    min_vol_index
]



max_weights = weights_list[
    max_sharpe_index
]


min_weights = weights_list[
    min_vol_index
]





# ==========================================================
# 9. Efficient Frontier Plot
# ==========================================================


plt.figure(
    figsize=(10,6)
)


plt.scatter(

    portfolio_results["Volatility"],

    portfolio_results["Return"],

    c=portfolio_results["Sharpe"],

    cmap="viridis",

    s=5

)


plt.colorbar(
    label="Sharpe Ratio"
)



plt.scatter(

    max_sharpe["Volatility"],

    max_sharpe["Return"],

    marker="*",

    color="red",

    s=300,

    label="Maximum Sharpe Portfolio"

)



plt.scatter(

    min_vol["Volatility"],

    min_vol["Return"],

    marker="X",

    color="blue",

    s=200,

    label="Minimum Volatility Portfolio"

)



plt.xlabel(
    "Portfolio Volatility"
)


plt.ylabel(
    "Expected Annual Return"
)


plt.title(
    "Efficient Frontier: TSLA + SPY + BND"
)


plt.legend()

plt.grid()

plt.show()





# ==========================================================
# 10. Portfolio Weights
# ==========================================================


portfolio_weights = pd.DataFrame({

    "Asset":
    [
        "TSLA",
        "SPY",
        "BND"
    ],


    "Maximum Sharpe Portfolio":
    max_weights,


    "Minimum Volatility Portfolio":
    min_weights

})



print("\nPortfolio Weights")

display(
    portfolio_weights
)





# ==========================================================
# 11. Final Metrics
# ==========================================================


print("\n========== MAXIMUM SHARPE PORTFOLIO ==========")


print(
    "Expected Annual Return:",
    round(max_sharpe["Return"]*100,2),
    "%"
)


print(
    "Annual Volatility:",
    round(max_sharpe["Volatility"]*100,2),
    "%"
)


print(
    "Sharpe Ratio:",
    round(max_sharpe["Sharpe"],3)
)





print("\n========== MINIMUM VOLATILITY PORTFOLIO ==========")


print(
    "Expected Annual Return:",
    round(min_vol["Return"]*100,2),
    "%"
)


print(
    "Annual Volatility:",
    round(min_vol["Volatility"]*100,2),
    "%"
)





# ==========================================================
# 12. Recommendation
# ==========================================================


recommended_weights = max_weights



print(
"""
FINAL PORTFOLIO RECOMMENDATION

The Maximum Sharpe Ratio Portfolio is selected as the optimal portfolio
because it provides the highest return per unit of risk. The portfolio
combines Tesla's forecast-based growth opportunity with SPY market
diversification and BND's defensive characteristics. This creates a
balanced allocation that improves risk-adjusted performance compared with
holding Tesla alone.
"""
)



print("\nRecommended Allocation:")


for asset, weight in zip(
    ["TSLA","SPY","BND"],
    recommended_weights
):

    print(
        asset,
        ":",
        round(weight*100,2),
        "%"
    )